## Ray core patterns and scheduling  
We will cover the following in this module:  
1. `ray.wait`        — don't block on all tasks, process as they finish
2. Pipelines       — chain tasks efficiently
3. Nested tasks    — tasks spawning tasks
4. Scheduling      — resource hints, placement groups
5. Fault tolerance — retries, actor recovery
6. Anti-patterns   — the most common mistakes (critical)

In [ ]:
# import all packages
import ray
from pprint import pprint
import time
import torch
import random

In [ ]:
context = ray.init(
    ignore_reinit_error=True, 
    include_dashboard=True, 
    dashboard_host="0.0.0.0"
)
print(context.dashboard_url)

In [ ]:
torch.cuda.is_available()

### 1. `ray.wait` usage  
`ray.get(refs)` on a full list has its own problem — it blocks until every single task finishes. If one task is slow, you wait for it before processing any results.
This is exactly where `ray.wait()` comes in.  
  
Key Differences:  
```mermaid  
ray.get(refs)     → waits for ALL → processes ALL at once at the end
ray.wait(refs)    → processes EACH as it finishes → lower latency
```  
  
**Pattern 1 — `ray.wait()`: Process results as they finish**

In [ ]:
@ray.remote
def process(x: int) -> int:
    time.sleep(random.uniform(0.5, 3))
    return x*2

In [ ]:
data = list(range(8))

refs = [process.remote(x) for x in data]

ready, remaining = ray.wait(refs, num_returns=1) # wait for atleast 1
print(f"The first result: {ray.get(ready)}")
print(f"Still running: {len(remaining)}")


The real power — process results as they arrive:

In [ ]:
data = list(range(8))
refs = [process.remote(x) for x in data]
remaining = refs.copy()
results = []

while remaining:
    # Wait for any ONE task to finish
    ready, remaining = ray.wait(remaining, num_returns=1)
    result = ray.get(ready[0])
    results.append(result)
    print(f"Got result: {result} | still waiting: {len(remaining)}")

print(f"\nAll results: {sorted(results)}")

Here are the `ray.wait` parameters  
```mermaid
ready, remaining = ray.wait(
    refs,               # list of ObjectRefs to wait on
    num_returns=1,      # how many to wait for before returning
    timeout=None,       # max seconds to wait (None = wait forever)
    fetch_local=True    # whether to fetch to local node
)
```

**Pattern 2 — Batch processing with ray.wait()**  
  
Real workloads often can't submit everything at once — millions of tasks would overwhelm the scheduler. Process in batches:

In [ ]:
@ray.remote
def process(x):
    time.sleep(0.1)
    return x*2

def process_in_batches(
    data: list[int],
    batch_size: int = 10
) -> list[int]:
    
    results = []

    # submit first batch
    refs = [process.remote(x) for x in data[:batch_size]]
    remaining_data = data[batch_size: ]

    while refs:

        # wait for one to finish
        ready, refs = ray.wait(refs, num_returns=1)
        result = ray.get(ready[0])
        results.append(result)

        # Submit next item as one finishes (sliding window)
        if remaining_data:
            refs.append(process.remote(remaining_data.pop(0)))

    return results


In [ ]:
data = list(range(50))
start = time.time()
results = process_in_batches(data, batch_size=10)
print(f"Processed {len(results)} items in {time.time()-start:.1f}s")
print(f"Sample: {sorted(results)[:5]}")

```mermaid
ray.get(refs)              → block until ALL done, return all results
ray.wait(refs, n)          → block until N done, return ready + remaining
ray.wait(..., timeout=t)   → block at most t seconds
sliding window pattern     → keep N tasks in flight, submit as each finishes
```

### 2. Nested Tasks  
#### What are Nested Tasks?  
A task spawning other tasks from inside itself. This lets you build dynamic parallel trees where work fans out automatically without the main process managing everything.  
```mermaid
Normal (main process manages all):     Nested (tasks manage tasks):

main ──► task0                         main ──► coordinator
main ──► task1                                      ├──► subtask0
main ──► task2                                      ├──► subtask1
main ──► task3                                      └──► subtask2
```

**Step 1 — Basic nested task**

In [ ]:
@ray.remote
def multiply(x, y):
    return x * y

@ray.remote
def sum_of_products(
    values: list[int], 
    multiplier: int
) -> int:
    # This task spawns subtasks from inside itself
    refs = [multiply.remote(v, multiplier) for v in values]
    products = ray.get(refs)
    return sum(products)

In [ ]:
# Main only submits one task
ref = sum_of_products.remote([1, 2, 3, 4, 5], 10)
print(ray.get(ref))  # 150

The main process submitted one task. That task internally spawned 5 more. Main doesn't know or care about those subtasks

**Step 2 — Recursive tree (divide and conquer)**  
  
The classic use case — parallel merge sort style reduction:

In [ ]:
@ray.remote
def parallel_sum(data: list[int]) -> int:

    # the base case - small enough, just sum it
    if len(data) <= 4:
        return sum(data)
    
    # Split and recurse — spawn two subtasks
    mid = len(data)//2
    left_refs = parallel_sum.remote(data[:mid])
    right_refs = parallel_sum.remote(data[mid:])

    # Wait for both halves and combine
    return sum(ray.get([left_refs, right_refs]))

In [ ]:
data = list(range(64))  # [0, 1, 2, ... 63]

result = ray.get(parallel_sum.remote(data))
print(f"Sum: {result}")           # 2016
print(f"Expected: {sum(data)}")   # 2016

The execution tree looks like:  
```mermaid
parallel_sum([0..63])
    ├── parallel_sum([0..31])
    │       ├── parallel_sum([0..15])
    │       │       ├── parallel_sum([0..7])  → sum
    │       │       └── parallel_sum([8..15]) → sum
    │       └── parallel_sum([16..31])
    │               ├── ...
    └── parallel_sum([32..63])
            └── ...
```  
All branches run in parallel automatically.

**Step 3 — Nested tasks with Actors**  
  
A coordinator actor managing a pool of worker tasks — very common pattern:

In [ ]:
@ray.remote
def heavy_work(task_id: int, duration: float) -> dict:
    time.sleep(duration)
    return {"task_id": task_id, "duration": duration}

@ray.remote
class Coordinator:
    def __init__(self):
        self.results = []
        self.failed  = []

    def run_batch(self, n_tasks: int) -> list[dict]:
        # Spawn tasks from inside the actor
        refs = [
            heavy_work.remote(i, random.uniform(0.1, 0.5))
            for i in range(n_tasks)
        ]

        # Process as they finish
        remaining = refs.copy()
        while remaining:
            ready, remaining = ray.wait(remaining, num_returns=1)
            result = ray.get(ready[0])
            self.results.append(result)

        return self.results

    def get_summary(self) -> dict:
        return {
            "total":   len(self.results),
            "avg_duration": sum(r["duration"] for r in self.results) / len(self.results)
        }

In [ ]:
# Main just talks to the coordinator
coordinator = Coordinator.remote()
results = ray.get(coordinator.run_batch.remote(8))
summary = ray.get(coordinator.get_summary.remote())

print(f"Completed: {summary['total']} tasks")
print(f"Avg duration: {summary['avg_duration']:.2f}s")

**Important gotcha: avoid deep nesting**  
  
Nested tasks are powerful but have a cost — each level adds scheduling overhead. This is a real anti-pattern:

Rule of thumb:  
```mermaid
Use nested tasks when:           Avoid nested tasks when:
─────────────────────            ────────────────────────
Work fans out dynamically        Simple linear pipeline
Divide and conquer               Nesting just for structure
Coordinator managing workers     Each level does trivial work
Unknown work size upfront        Flat submission works fine
```

### 3. Scheduling & Resources  
#### What is Ray Scheduling?
When you submit a task or create an actor, Ray's scheduler decides which worker process on which node runs it. By default Ray handles this automatically, but for GPU training you need explicit control — you want to tell Ray exactly which resources a task needs.

In [ ]:
# Tell Ray this task needs 1 CPU (default)
@ray.remote(num_cpus=1)
def cpu_task(x):
    return x * 2

# Tell Ray this task needs 1 GPU
@ray.remote(num_gpus=1)
def gpu_task(x):
    import torch
    device = torch.cuda.current_device()
    print(f"Running on GPU: {torch.cuda.get_device_name(device)}")
    return x * 2

In [ ]:
# Check what your cluster has available
print(ray.cluster_resources())
# {'CPU': 16.0, 'GPU': 1.0, 'memory': ...}

# Check what's currently free
print(ray.available_resources())

**Step 2 — Fractional resources**

In [ ]:
@ray.remote(num_gpus=0.25)
def small_gpu_task(task_id: int) -> str:
    import torch
    # All tasks share the same physical GPU
    # Ray sets CUDA_VISIBLE_DEVICES for isolation
    return f"Task {task_id} on GPU {torch.cuda.current_device()}"

In [ ]:
# 4 tasks run concurrently on your single GPU
refs = [small_gpu_task.remote(i) for i in range(4)]
print(ray.get(refs))

**Step 3 — Resource requirements on Actors**

In [ ]:
@ray.remote(num_gpus=1, num_cpus=4)
class GPUTrainer:
    def __init__(self, model_name: str):
        import torch
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model_name = model_name
        print(f"Trainer initialized on {self.device}")

    def train_step(self, batch: list) -> dict:
        import torch
        # Simulate a training step
        loss = torch.tensor(float(len(batch))).to(self.device)
        return {
            "model":  self.model_name,
            "loss":   loss.item(),
            "device": str(self.device)
        }

    def get_device(self) -> str:
        return str(self.device)

In [ ]:
# This actor owns 1 GPU for its lifetime
trainer = GPUTrainer.remote("my-transformer")
print(ray.get(trainer.get_device.remote()))

# Run some training steps
for i in range(3):
    result = ray.get(trainer.train_step.remote(list(range(32))))
    print(f"Step {i}: {result}")

**Step 4 — Placement Groups**  
  
This is the most important scheduling concept for multi-GPU training. A placement group reserves a bundle of resources atomically — either all resources are available and reserved together, or nothing is reserved.  
```mermaid
Without placement groups:           With placement groups:
────────────────────────            ──────────────────────
worker0 gets GPU 0 ✓                Reserve GPU 0 + GPU 1 atomically
worker1 waits for GPU 1...          Only start when BOTH are available
training starts inconsistently      All workers start together ✓
```

In [ ]:
from ray.util.placement_group import placement_group
from ray.util.scheduling_strategies import PlacementGroupSchedulingStrategy

In [ ]:
ray.shutdown()

In [ ]:
# Reserve 1 bundle: 1 GPU + 2 CPUs — atomically
pg = placement_group(
    bundles=[{"GPU": 1, "CPU": 2}],
    strategy="STRICT_PACK"   # all bundles on same node
)

# Wait until resources are actually reserved
ray.get(pg.ready())
print("Placement group ready!")

In [ ]:
print(ray.cluster_resources())
print(ray.available_resources())

In [ ]:
@ray.remote(num_gpus=1, num_cpus=2)
class TrainingWorker:
    def __init__(self, worker_id: int):
        import torch
        self.worker_id = worker_id
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def train(self, steps: int) -> dict:
        import torch
        losses = []
        for step in range(steps):
            # Simulate training
            loss = torch.rand(1).item()
            losses.append(loss)
        return {
            "worker_id": self.worker_id,
            "avg_loss":  sum(losses) / len(losses),
            "device":    str(self.device)
        }

Placement Group strategy  

```python
# STRICT_PACK — all bundles on the same node (best for single machine)
pg = placement_group(bundles=[...], strategy="STRICT_PACK")

# PACK — prefer same node, allow spillover (default)
pg = placement_group(bundles=[...], strategy="PACK")

# SPREAD — force bundles onto different nodes (multi-node training)
pg = placement_group(bundles=[...], strategy="SPREAD")
```

**Step 5 — Multi-worker setup (preview of Ray Train)**  
  
This is exactly what Ray Train does under the hood:

```mermaid
num_cpus / num_gpus       → declare resource needs per task/actor
fractional GPU            → concurrent sharing (you manage memory)
ray.cluster_resources()   → total cluster resources
ray.available_resources() → currently free resources
placement_group           → atomic resource reservation
pg.ready()                → wait until all resources reserved
STRICT_PACK               → all on same node (single machine)
SPREAD                    → across different nodes (multi-node)
PlacementGroupSchedulingStrategy → pin actor/task to specific bundle
```